# 03 — Training

**Auralytics** | CPEN 355 Final Project

Trains the convolutional autoencoder on normal clips for one machine type.
Designed to run on **Google Colab** (free GPU tier).

At the end of this notebook you will have:
- a trained `.pth` checkpoint in `models/`
- a loss curve saved to `results/`
- a quick reconstruction preview to verify the model learned something


## 0. Colab Setup

If running on Colab, mount Drive and clone the repo first:

```python
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/YOUR_USERNAME/auralytics.git
%cd auralytics
!pip install -r requirements.txt -q
```

Then upload your `data/processed/` folder to the repo root on Colab (or sync from Drive).

If running **locally**, skip this cell.

In [ ]:
# ── Portable repo-root resolution (same pattern as 01_eda.ipynb) ─────────────
import sys
from pathlib import Path

def _find_repo_root(marker: str = 'src') -> Path:
    candidate = Path.cwd()
    for _ in range(6):
        if (candidate / marker).is_dir():
            return candidate
        candidate = candidate.parent
    raise RuntimeError(f"Could not find repo root ('{marker}/' not found).")

REPO_ROOT = _find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f'Repo root: {REPO_ROOT}')

In [ ]:
import torch
import matplotlib.pyplot as plt

from src.model import ConvAutoencoder
from src.train import train_model
from src.dataset import make_dataloaders
from src.utils import get_device, load_checkpoint, plot_reconstruction

PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
MODELS_DIR    = REPO_ROOT / 'models'
RESULTS_DIR   = REPO_ROOT / 'results'

device = get_device()
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

## 1. Choose Machine Type
Set this and re-run to train a different model.

In [ ]:
MACHINE = 'fan'   # 'fan' | 'pump' | 'valve'

## 2. Quick Model Inspection

In [ ]:
model = ConvAutoencoder(base_ch=32)
print(model)

# Verify forward pass with a dummy input
dummy = torch.randn(4, 1, 128, 313)   # batch=4, 1 channel, 128 mels, ~313 time frames
recon = model(dummy)
print(f'Input shape  : {tuple(dummy.shape)}')
print(f'Output shape : {tuple(recon.shape)}')
assert recon.shape == dummy.shape, 'Shape mismatch — check model!'
print('✓ Shape check passed')

## 3. Train
Early stopping is enabled — training halts automatically if val loss plateaus.

In [ ]:
train_model(
    machine_type  = MACHINE,
    processed_dir = PROCESSED_DIR,
    models_dir    = MODELS_DIR,
    results_dir   = RESULTS_DIR,
    epochs        = 50,
    batch_size    = 32,
    lr            = 1e-3,
    patience      = 8,
)

## 4. Load Best Checkpoint and Preview Reconstructions

In [ ]:
model = ConvAutoencoder(base_ch=32).to(device)
load_checkpoint(MODELS_DIR / f'{MACHINE}_best.pth', model, device=device)

_, _, test_loader = make_dataloaders(PROCESSED_DIR, MACHINE, batch_size=8)

# Grab one batch from the test set
specs, labels = next(iter(test_loader))
specs = specs.to(device)

with torch.no_grad():
    recons = model(specs)
    scores = model.anomaly_score(specs)

# Show first normal and first anomalous reconstruction
for target_label, name in [(0, 'Normal'), (1, 'Anomalous')]:
    idx = next((i for i, l in enumerate(labels) if l == target_label), None)
    if idx is None:
        print(f'No {name} clip in this batch — re-run cell or increase batch_size')
        continue
    plot_reconstruction(
        original      = specs[idx].cpu().numpy(),
        reconstructed = recons[idx].cpu().numpy(),
        anomaly_score = scores[idx].item(),
        label         = labels[idx].item(),
    )

## ✅ Training Complete

Checkpoint saved to `models/{machine}_best.pth`.

What to check:
- Normal clips should have **low** anomaly scores
- Anomalous clips should have **high** anomaly scores
- The error map should highlight the regions that differ

**Next:** `04_evaluation.ipynb` — AUC, pAUC, F1, and score distributions.